# Tool Calling, End to End

**For people who have never seen this before.**

Goal of this notebook: by the end, you will know exactly what happens when an LLM "calls a tool." There is no magic. It is just JSON going in and JSON coming out, plus a Python `if` statement that runs a real function.

We will go in this order:

1. Send a plain message to an LLM and look at the raw response.
2. Bind a single tool and look at how the response changes.
3. Inspect the JSON the model emits when it wants to call a tool.
4. Bind multiple tools and trigger parallel tool calls.
5. Change the tool description and watch the model's behavior change.
6. Write the loop that actually runs the tool and feeds the result back.

We use plain `requests` for any external API so nothing feels like a black box.

## 0. Install dependencies

Run this once. We use the latest LangChain split packages.

In [1]:
# Dependencies are already declared in pyproject.toml and installed via:
#   uv add ...
# If you are running this notebook with the project's uv venv as the
# kernel, the imports below will just work. If not, run this cell.
#
# uv-created venvs do NOT ship pip, so %pip install fails with
# 'No module named pip'. Use uv pip instead:
!uv pip install -q "langchain>=0.3" "langchain-openai>=0.3" "langchain-core>=0.3" requests python-dotenv

# If you are on a regular venv with pip, swap to:
# %pip install -q "langchain>=0.3" "langchain-openai>=0.3" "langchain-core>=0.3" requests python-dotenv

## 1. Set your API key

Put your OpenAI key in a `.env` file in this folder:

```
OPENAI_API_KEY=sk-...
```

If you want to use the search tool later, also add:

```
TAVILY_API_KEY=tvly-...
```

Tavily has a free tier. Sign up at https://tavily.com. If you do not have one, the search cell will fall back to a fake search so the lesson still works.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("OpenAI key loaded:", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("Tavily key loaded:", "yes" if os.getenv("TAVILY_API_KEY") else "no (will use a fake search)")

## 2. The simplest possible LLM call

Before we touch tools, look at what happens when we send one message and get one message back. This is the baseline.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

response = llm.invoke([HumanMessage(content="What is 2 + 2?")])

print(response)
print("Type of response:", type(response).__name__)
print()
print("response.content:")
print(response.content)
print()
print("response.tool_calls:")
print(response.tool_calls)

**Read the output carefully.**

- `response.content` is a string. The model wrote text.
- `response.tool_calls` is an empty list. The model did not ask to run any tool.

That is the entire shape of an LLM response in LangChain. Text in, text out. Now we add tools.

## 3. Define a tool

A tool is just a Python function with a name, a description, and typed arguments. We use the `@tool` decorator. The docstring becomes the description the model sees.

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b

print("Tool name:       ", add.name)
print("Tool description:", add.description)
print("Tool args schema:", add.args)

Notice three things the model will see:

1. The **name** (`add`).
2. The **description** (the docstring).
3. The **argument schema** (`a: int`, `b: int`).

That is all the model knows about your tool. It picks tools based on these three things.

## 4. Bind the tool to the LLM and ask the same question

We use `.bind_tools([add])`. This does **not** run the tool. It just tells the LLM "these tools exist, here is their schema."

In [ ]:
llm_with_tools = llm.bind_tools([add])

response = llm_with_tools.invoke([HumanMessage(content="What is 17 + 25?")])

print("response.content:")
print(repr(response.content))
print()
print("response.tool_calls:")
for tc in response.tool_calls:
    print(tc)

**This is the moment to slow down.**

- `response.content` is now an empty string. The model did not write text. It chose to call a tool instead.
- `response.tool_calls` contains one entry. It has a `name`, `args`, and an `id`.

The model literally said: "do not show text to the user, instead call `add` with `a=17` and `b=25`."

**The model did not run the tool. We have to.** That is the whole trick. The model only outputs the request.

## 5. Look at the actual JSON sent over the wire

Let's peek at the raw request and response so there is no mystery.

In [ ]:
import json
from langchain_core.utils.function_calling import convert_to_openai_tool

openai_tool_spec = convert_to_openai_tool(add)
print("This is exactly what gets sent to OpenAI as the tool definition:")
print(json.dumps(openai_tool_spec, indent=2))

In [ ]:
print("And this is the raw response that came back:")
print("response.tool_calls:")
print(json.dumps(response.tool_calls, indent=2, default=str))
print()
print("response.additional_kwargs (raw provider data):")
print(json.dumps(response.additional_kwargs, indent=2, default=str))

Read those two JSON blocks. That is everything. The model gets a list of tool schemas, and emits a `tool_calls` array. There is no other channel.

## 6. Run the tool the model asked for

Now we close the loop. Take the tool call, look up the function, run it, send the result back.

In [ ]:
from langchain_core.messages import ToolMessage

tool_call = response.tool_calls[0]
print("Model requested:", tool_call["name"], "with", tool_call["args"])

tool_registry = {"add": add}
selected = tool_registry[tool_call["name"]]
result = selected.invoke(tool_call["args"])
print("Tool returned: ", result)

messages = [
    HumanMessage(content="What is 17 + 25?"),
    response,
    ToolMessage(content=str(result), tool_call_id=tool_call["id"]),
]

final = llm_with_tools.invoke(messages)
print()
print("Final answer from model:")
print(final.content)

Walk through the conversation we just built:

1. `HumanMessage` ("What is 17 + 25?")
2. `AIMessage` (the model's tool call request, no text)
3. `ToolMessage` (our function's return value, tagged with the same `tool_call_id`)
4. `AIMessage` (now the model sees the result and writes a text answer)

The `tool_call_id` is the glue. It tells the model "this result corresponds to that earlier request."

## 7. Multiple tools and parallel tool calls

Real agents have many tools. We add a few and then ask a question that requires more than one. Modern OpenAI models can request multiple tool calls in a single response. This is parallel tool calling.

In [ ]:
import requests

@tool
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the product."""
    return a * b

@tool
def web_search(query: str) -> str:
    """Search the public web for current information. Use this for facts that change over time, like news, prices, weather, sports scores."""
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return f"[FAKE SEARCH] No Tavily key set. Pretend results for: {query}"
    r = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": api_key, "query": query, "max_results": 3},
        timeout=15,
    )
    r.raise_for_status()
    data = r.json()
    return "\n".join(f"- {item['title']}: {item['content'][:200]}" for item in data.get("results", []))

tools = [add, multiply, web_search]
tool_registry = {t.name: t for t in tools}

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke([
    HumanMessage(content="Compute 12 * 7 and also 100 + 250.")
])

print(f"Number of tool calls in one response: {len(response.tool_calls)}")
for tc in response.tool_calls:
    print(" ", tc["name"], tc["args"])

Two tool calls came back in a single response. The model decided both are independent so it can run them in parallel.

Now we run both, send both results back, and let the model write the final answer.

In [ ]:
messages = [
    HumanMessage(content="Compute 12 * 7 and also 100 + 250."),
    response,
]

for tc in response.tool_calls:
    fn = tool_registry[tc["name"]]
    result = fn.invoke(tc["args"])
    print(f"Ran {tc['name']}({tc['args']}) -> {result}")
    messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

final = llm_with_tools.invoke(messages)
print()
print("Final answer:")
print(final.content)

## 8. Tool descriptions matter, a lot

The description is how the model decides whether to use a tool. Change the description, change the behavior. Watch this.

In [ ]:
@tool
def lookup_user(user_id: str) -> str:
    """Get information about a user."""
    return f"User {user_id}: name=Alice, plan=Pro"

vague_llm = llm.bind_tools([lookup_user])
resp = vague_llm.invoke([HumanMessage(content="How many users do we have?")])
print("With VAGUE description:")
print("  content:    ", repr(resp.content)[:120])
print("  tool_calls: ", resp.tool_calls)

In [ ]:
@tool
def lookup_user(user_id: str) -> str:
    """Look up ONE user by their exact user_id. Returns name and plan. Do NOT use for counting users or listing users. Only use when the caller already has a specific user_id."""
    return f"User {user_id}: name=Alice, plan=Pro"

precise_llm = llm.bind_tools([lookup_user])
resp = precise_llm.invoke([HumanMessage(content="How many users do we have?")])
print("With PRECISE description:")
print("  content:    ", resp.content[:200])
print("  tool_calls: ", resp.tool_calls)

Same function, same name, same arguments. Only the docstring changed.

With the vague description, the model may try to use the tool incorrectly. With the precise description that explains *when not to use it*, the model declines and asks for clarification.

**Lesson for your students:** writing tool descriptions is prompt engineering. Be specific. Tell the model when to use the tool and when not to.

## 9. The full agent loop

Now we put it all together. The agent loop is just:

```
while True:
    response = llm.invoke(messages)
    if no tool_calls in response: break
    for each tool_call: run it, append result
```

That is it. No framework. No magic.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

def run_agent(user_input: str, max_iters: int = 6):
    messages = [HumanMessage(content=user_input)]
    for step in range(1, max_iters + 1):
        print(f"\n----- iteration {step} -----")
        ai_msg: AIMessage = llm_with_tools.invoke(messages)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:
            print("No tool calls. Final answer:")
            print(ai_msg.content)
            return ai_msg.content

        print(f"Model requested {len(ai_msg.tool_calls)} tool call(s):")
        for tc in ai_msg.tool_calls:
            fn = tool_registry[tc["name"]]
            result = fn.invoke(tc["args"])
            print(f"  {tc['name']}({tc['args']}) -> {str(result)[:120]}")
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    print("Hit max iterations without final answer.")
    return None

run_agent("What is 23 * 4, and then add 100 to that result?")

Notice something important in the output above: the model called `multiply` first, got the result, then on the next iteration called `add` with the previous result. It chained tool calls across iterations because the second call needed the first one's output.

In [ ]:
run_agent("Search the web for the current price of bitcoin and tell me if it is above 50000.")

## What you now know

1. An LLM response has two parts: `content` (text) and `tool_calls` (a list of structured requests).
2. A tool definition is a name, a description, and an argument schema. That is it.
3. The model never executes anything. It only emits requests. You run the function.
4. You feed results back as `ToolMessage` with the matching `tool_call_id`.
5. Modern models can request multiple tools in one response (parallel tool calls).
6. Tool descriptions are prompts. Sloppy descriptions lead to sloppy tool use.
7. The agent loop is `while`: keep calling the model until it stops asking for tools.

Next notebook: `02_react_loop.ipynb` shows the same loop done a different way, using the original ReAct paper's prompting style with the prompt fetched from LangChain Hub.